# Building upon the model and evaluate method used in InsectNet
https://academic.oup.com/pnasnexus/article/4/1/pgae575/7933354?login=false

### Imports

In [1]:
import sys
import csv
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
import torchvision
from PIL import Image as PILImage
from PIL.ExifTags import TAGS


# Resolve notebook directory robustly — works even if kernel CWD differs
_nb_dir = Path(globals().get("__vsc_ipynb_file__", "")).parent
if not _nb_dir.is_dir():
    _nb_dir = Path.cwd()
sys.path.insert(0, str(_nb_dir / "InsectNet"))
from evaluate import evaluate

PROJECT_DIR = _nb_dir

### Configuration

All tunable parameters are here. Override any of them when calling `main()` without touching the rest of the code.

Parameters marked **TUNE** are the ones most likely to need adjustment per dataset or plot.

In [2]:
DEFAULT_CONFIG = {

    # Background computation
    # Number of frames to sample for median background. TUNE
    # Set to 0 or None to skip global background entirely (use rolling_window instead).
    # Set to 0 + rolling_window > 0 = pure rolling/frame-to-frame mode (no memory overhead).
    "background_sample_size":    0,

    # Background subtraction
    # How much darker/lighter than the median a pixel must be to count as foreground.
    # Lower = more sensitive (more detections, more noise).
    # Higher = less sensitive (fewer detections, fewer false positives). TUNE
    "darker_threshold":          30,   # lowered — detect colour-similar insects (yellow/black)

    # Contour filtering
    "min_contour_area":          400,   # lowered — small insects (area ~495) were filtered
    "max_contour_area":          30000,  # upper limit to filter merged noise blobs
    "max_aspect_ratio":          5,    # raised — parasitoid wasps are very elongated
    "kernel_open_size":          3,    # morphological open kernel — removes isolated speckles
    "kernel_close_size":         11,   # morphological close kernel — merges insect body fragments. TUNE
    # Minimum grayscale std dev inside contour (insects have texture, soil does not). TUNE
    "min_texture":               50,

    # Static detection removal
    "static_dist":               80,
    "static_max_frames":         15,

    # Crop padding (proportional — 10% of bbox size)
    "padding_ratio":             0.1,

    # Rolling background window. TUNE
    # 0 = use global median background (default)
    # 1 = pure frame-to-frame diff (no memory, misses stationary insects)
    # N > 1 = median of last N frames (bumblebee visible if it moves within N frames)
    # Tip: set background_sample_size=0 to skip building global background entirely
    "rolling_window": 1,

    # Marker / ROI
    "marker_hue":                (45, 75),
    "marker_sat_min":            200,
    "marker_val_min":            100,
    "marker_min_area":           200,
    "marker_zone_radius":        800,

    # Quality filtering
    "skip_flash":                True,   # skip frames where flash fired (night shots). TUNE
    "skip_foggy":                True,   # skip frames below foggy_threshold. TUNE
    "foggy_threshold":           50,    # Laplacian variance below this = fog/blur. TUNE

    # Weather classification from EXIF shutter speed
    "sunny_shutter_threshold":   150,

    # Near-flower detection
    "near_flower_iou_threshold": 0.1,

    # Pollinator class mapping is handled by map_to_broad_class()
    # using species/common name for conservative classification.

    # Manual ROI — set True to draw ROI manually instead of auto-detecting marker
    # Useful for cameras where the colored marker is not visible
    "manual_roi": False,

    # Skip InsectNet inference — only save crops, do not classify.
    # Use this when tuning detection parameters to speed up the pipeline 10x.
    "skip_insectnet": False,
}

CSV_FIELDS_DEBUG = [
    "image_name", "crop_filename", "datetime", "camera_name",
    "shutter_speed", "weather",
    "skip", "skip_reason", "laplacian_var",
    "pollinator_detected", "pollinator_type",
    "scientific_name", "common_name",
    "order", "family",
    "insectnet_confirmed", "confidence", "energy_score",
    "near_marked_flower",
    "detection_scope",
    "bbox_x", "bbox_y", "bbox_w", "bbox_h",
]

CSV_FIELDS_MARIA = [
    "image_name", "crop_filename", "datetime", "camera_name",
    "shutter_speed", "weather",
    "pollinator_detected", "pollinator_type",
    "scientific_name",
    "common_name",
    "order",
    "confidence", "energy_score",
    "near_marked_flower",
    "detection_scope",
]


### Helper functions

In [3]:

from pathlib import Path

def get_relative_path(path, root):
    return Path(path).relative_to(root)

def ensure_dir(p):
    p.mkdir(parents=True, exist_ok=True)
import re
from PIL import Image as PILImageSort
from PIL.ExifTags import TAGS as TAGS_SORT

# Find EXIF DateTimeOriginal tag ID
_EXIF_DT_TAG = next((k for k, v in TAGS_SORT.items() if v == "DateTimeOriginal"), None)
_exif_cache: dict = {}

def get_exif_datetime_sort(path) -> str | None:
    """Read EXIF DateTimeOriginal from image, with caching."""
    key = str(path)
    if key in _exif_cache:
        return _exif_cache[key]
    try:
        img = PILImageSort.open(path)
        exif = img._getexif() if hasattr(img, "_getexif") else dict(img.getexif())  # type: ignore[attr-defined]
        val = exif.get(_EXIF_DT_TAG) if exif and _EXIF_DT_TAG else None
    except Exception:
        val = None
    _exif_cache[key] = val
    return val

def _parse_exif_dt(dt_str: str):
    """Parse '2023:07:14 12:34:56' into a sortable tuple."""
    try:
        date, time = dt_str.split(" ")
        y, m, d = map(int, date.split(":"))
        hh, mm, ss = map(int, time.split(":"))
        return (y, m, d, hh, mm, ss)
    except Exception:
        return None

def robust_sort_key(p):
    """Sort by: 1) EXIF DateTimeOriginal, 2) filename number, 3) filename string.

    Ensures correct temporal ordering for background subtraction and
    frame-to-frame difference computation, even across camera roll boundaries.
    """
    from pathlib import Path
    p = Path(p)
    exif_dt = get_exif_datetime_sort(p)
    if exif_dt:
        parsed = _parse_exif_dt(exif_dt)
        if parsed:
            return (0, parsed, p.name)
    m = re.search(r'(\d+)', p.stem)
    if m:
        return (1, int(m.group(1)), p.name)
    return (2, p.name)


In [ ]:
# ══════════════════════════════════════════════════════════════════
# MODEL
# ══════════════════════════════════════════════════════════════════

def load_model():
    """Load InsectNet RegNet weights from model.pth."""
    weights = torch.load(
        PROJECT_DIR / "model.pth",
        map_location=torch.device("cpu"),
        weights_only=False,
    )["model"]
    model = torchvision.models.regnet_y_32gf()
    model.fc = torch.nn.Linear(3712, 2526)
    model.load_state_dict(weights, strict=True)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    model.eval()
    return model


def load_model_and_classes():
    """Step 6: Load InsectNet model and class metadata."""
    print("Loading model...")
    model = load_model()
    cmn_df = pd.read_csv(PROJECT_DIR / "InsectNet" / "data" / "classes.csv")
    class_txt_path = str(PROJECT_DIR / "InsectNet" / "data" / "classes.txt")
    print("Model loaded.\n")
    return model, cmn_df, class_txt_path


# ══════════════════════════════════════════════════════════════════
# ROI / ZONE
# ══════════════════════════════════════════════════════════════════

def find_marker(image, cfg):
    """Find the green marker clip in the image. Returns centroid (cx, cy) or None."""
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    mask = cv2.inRange(
        hsv,
        np.array([cfg["marker_hue"][0], cfg["marker_sat_min"], cfg["marker_val_min"]]),
        np.array([cfg["marker_hue"][1], 255, 255]),
    )
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    clusters = [c for c in contours if cv2.contourArea(c) > cfg["marker_min_area"]]
    if not clusters:
        return None
    h_img, w_img = image.shape[:2]
    cx_img, cy_img = w_img // 2, h_img // 2

    def dist_to_center(c):
        M = cv2.moments(c)
        if M["m00"] == 0:
            return float("inf")
        return (int(M["m10"]/M["m00"]) - cx_img)**2 + (int(M["m01"]/M["m00"]) - cy_img)**2

    best = min(clusters, key=dist_to_center)
    M = cv2.moments(best)
    return int(M["m10"] / M["m00"]), int(M["m01"] / M["m00"])


def build_marker_zone(image, marker, cfg):
    """Create a circular binary mask (zone) around the marker position."""
    h_img, w_img = image.shape[:2]
    zone = np.zeros((h_img, w_img), dtype=np.uint8)
    cv2.circle(zone, marker, cfg["marker_zone_radius"], 255, -1)
    return zone


def select_roi(image_path):
    """Open the first image and let user draw a rectangle as the watching zone."""
    img = cv2.imread(image_path)
    h, w = img.shape[:2]
    scale = min(1.0, 1200 / w)
    display = cv2.resize(img, (int(w * scale), int(h * scale)))
    print("Draw a rectangle around the flower area, then press ENTER or SPACE.")
    roi = cv2.selectROI("Select flower region", display, showCrosshair=True)
    cv2.destroyAllWindows()
    x, y, rw, rh = [int(v / scale) for v in roi]
    if rw == 0 or rh == 0:
        return None
    zone = np.zeros((h, w), dtype=np.uint8)
    zone[y:y+rh, x:x+rw] = 255
    print(f"ROI selected: x={x} y={y} w={rw} h={rh}")
    return zone


def setup_roi(paths, cfg, manual_roi):
    """Step 1: Define ROI from green marker, manual selection, or full image fallback.

    Priority:
    1. manual_roi=True → draw ROI manually (for cameras without visible marker)
    2. Auto-detect green marker → build circular zone around it
    3. No marker found → fallback to full image (no popup, runs unattended)
    """
    first_img = cv2.imread(str(paths[0]))
    if manual_roi:
        zone = select_roi(str(paths[0]))
        if zone is None:
            print("No region selected — falling back to full image as ROI.")
            zone = np.ones(first_img.shape[:2], dtype=np.uint8) * 255
        return zone, None
    marker = find_marker(first_img, cfg)
    if marker is None:
        print("No marker found — using full image as ROI (set manual_roi=True to draw manually)")
        zone = np.ones(first_img.shape[:2], dtype=np.uint8) * 255
        return zone, None
    print(f"Marker found at ({marker[0]}, {marker[1]})")
    zone = build_marker_zone(first_img, marker, cfg)
    pct = 100 * np.count_nonzero(zone) / zone.size
    print(f"Watching zone covers {pct:.1f}% of image")
    return zone, marker


# ══════════════════════════════════════════════════════════════════
# BACKGROUND
# ══════════════════════════════════════════════════════════════════

def build_background(paths, cfg):
    """
    Step 2: Compute median background from a uniform sample of frames.

    Uses cfg['background_sample_size'] frames evenly spaced across the sequence.
    Avoids loading all 12,000 frames into memory (~60GB) while producing
    a background equivalent to the full-dataset median.
    Per-pixel median naturally excludes transient objects like insects (~4% of frames).
    """
    n = cfg["background_sample_size"]
    # Global background is always needed as fallback for first frames
    # and as one of the two diff signals in use_prev_frame mode.
    if not n:
        sampled = list(paths)
    else:
        step = max(1, len(paths) // n)
        sampled = list(paths)[::step][:n]
    frames = [cv2.imread(str(p)) for p in sampled]
    frames = [f for f in frames if f is not None]
    if not frames:
        return None
    print(f"Background computed from {len(frames)} sampled frames (of {len(paths)} total)")
    return np.median(frames, axis=0).astype(np.uint8)


# ══════════════════════════════════════════════════════════════════
# DETECTION (background subtraction + contour filtering)
# ══════════════════════════════════════════════════════════════════

def detect_visitor(image, background, zone, cfg):
    """
    Find candidate insect regions in one frame by comparing to background.

    Strategy:
    1. Subtract background — pixels darker than median are potential foreground
    2. Exclude green vegetation (grass, leaves)
    3. Restrict to ROI zone
    4. Filter contours by area, aspect ratio, and texture
       - Too small  → noise
       - Too elongated → grass stems
       - No texture → (filter removed, handled by classifier)

    Returns list of bounding boxes [(x, y, w, h), ...].
    Note: these are candidates only; InsectNet classifies them in Step 7.
    """
    gray_img = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    gray_bg  = cv2.cvtColor(background, cv2.COLOR_BGR2GRAY)

    # Only compute diff within ROI zone — reduces noise from outside
    gray_img_roi = cv2.bitwise_and(gray_img, gray_img, mask=zone)
    gray_bg_roi  = cv2.bitwise_and(gray_bg,  gray_bg,  mask=zone)

    diff     = cv2.absdiff(gray_bg_roi, gray_img_roi)
    diff     = cv2.GaussianBlur(diff, (7, 7), 0)
    _, mask  = cv2.threshold(diff, cfg["darker_threshold"], 255, cv2.THRESH_BINARY)
    mask     = cv2.bitwise_and(mask, zone)

    # Exclude green vegetation
    hsv   = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    green = cv2.inRange(hsv, np.array([25, 40, 40]), np.array([95, 255, 255]))
    mask  = cv2.bitwise_and(mask, cv2.bitwise_not(green))

    # Morphological cleanup
    # OPEN removes isolated speckles; CLOSE merges nearby insect body fragments.
    # Increase kernel_close_size to merge more fragments; decrease to avoid merging background.
    ko = cfg.get("kernel_open_size",  3)
    kc = cfg.get("kernel_close_size", 11)
    kernel_open  = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (ko, ko))
    kernel_close = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (kc, kc))
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN,  kernel_open)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel_close)

    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    valid = []
    for c in contours:
        area = cv2.contourArea(c)
        if area < cfg["min_contour_area"]:
            continue
        if area > cfg.get("max_contour_area", 50000):
            continue
        x, y, w, h = cv2.boundingRect(c)
        if max(w, h) / max(min(w, h), 1) > cfg["max_aspect_ratio"]:
            continue

        # Texture filter removed — low-texture insects (e.g. blurry flies) were
        # being incorrectly filtered. False positives handled by InsectNet instead.
        valid.append((area, cv2.boundingRect(c)))

    valid.sort(key=lambda v: v[0], reverse=True)
    return [bbox for _, bbox in valid]


def detect_all_frames(paths, background, zone, cfg, debug=False, debug_dir=None, image_dir=None):
    """
    Step 4: Run background subtraction on every frame.

    If cfg['rolling_window'] > 0, uses a rolling median background instead of
    the global median, further reducing slow lighting-change false positives.
    """
    print("Detecting visitors...")
    all_detections = {}
    window       = cfg.get("rolling_window", 0)
    frames_cache = []  # for rolling background

    total = len(paths)
    for i, path in enumerate(paths):
        image = cv2.imread(str(path))
        if image is None:
            continue

        # Choose background: rolling (last N frames) or global median
        if window > 0 and len(frames_cache) >= 1:
            # Rolling background: grass that moves consistently across recent
            # frames is absorbed into the background and will not be detected.
            bg = np.median(frames_cache[-window:], axis=0).astype(np.uint8)
        else:
            # First frame (rolling) or global mode: use precomputed global background
            # Do NOT skip — skipping causes key mismatch in classify_and_write
            bg = background

        img_to_detect = image

        if debug and debug_dir:
            save_debug_images(
                str(Path(path).relative_to(image_dir)).replace("/", "_").replace("\\", "_").rsplit(".", 1)[0]
                if image_dir else path.stem,
                image, bg, zone, None, debug_dir, cfg)

        all_detections[str(path)] = detect_visitor(img_to_detect, bg, zone, cfg)

        # Update histories
        frames_cache.append(image)
        if window > 0 and len(frames_cache) > window:
            frames_cache.pop(0)

    return all_detections


def filter_static_detections(all_detections, cfg):
    """
    Step 5: Remove detections at the same position across too many frames.

    Real visitors appear in a few consecutive frames then leave.
    Soil patches, fixed shadows, and other static artefacts persist across
    many frames and are rejected here.

    static_max_frames is set to 15 (was 2 in original):
    bumblebees can stay on a flower for several minutes (~5 frames at 1-min intervals).
    """
    all_centers = [
        (x + w//2, y + h//2, path)
        for path, bboxes in all_detections.items()
        for x, y, w, h in bboxes
    ]

    static_positions = set()
    for cx, cy, _ in all_centers:
        frames_nearby = {p for cx2, cy2, p in all_centers
                         if ((cx-cx2)**2 + (cy-cy2)**2)**0.5 < cfg["static_dist"]}
        if len(frames_nearby) > cfg["static_max_frames"]:
            static_positions.add((cx, cy))

    if static_positions:
        print(f"Filtered {len(static_positions)} static position(s) (appeared in >{cfg['static_max_frames']} frames)")

    filtered = {}
    for path, bboxes in all_detections.items():
        filtered[path] = [
            (x, y, w, h) for x, y, w, h in bboxes
            if not any(((x+w//2-sx)**2 + (y+h//2-sy)**2)**0.5 < cfg["static_dist"]
                       for sx, sy in static_positions)
        ]
    return filtered


def crop_with_padding(image, bbox, cfg):
    """Crop a bounding box with proportional padding (10% of bbox size by default).
    Keeps crop tight around the insect without including excessive background.
    """
    x, y, w, h = bbox
    h_img, w_img = image.shape[:2]
    pad = max(5, int(max(w, h) * cfg.get("padding_ratio", 0.1)))
    return image[max(0,y-pad):min(h_img,y+h+pad), max(0,x-pad):min(w_img,x+w+pad)]


# ══════════════════════════════════════════════════════════════════
# WEATHER (EXIF)
# ══════════════════════════════════════════════════════════════════

def get_exif_metadata(image_path, cfg):
    """
    Step 3: Extract metadata, classify weather, and detect flash/fog.

    Camera: Wingscapes TLCAM PRO — fixed aperture f/2.8, auto exposure.
    Shutter speed reflects ambient light:
      sunny  → fast shutter → denominator > sunny_shutter_threshold
      cloudy → slow shutter → denominator <= sunny_shutter_threshold

    Flash detection: EXIF tag 37385 (Flash). Nonzero = flash fired → night/indoor shot.
    Foggy detection: Laplacian variance of grayscale image. Low variance = low contrast → fog/blur.

    Returns metadata dict with a "skip" key and "skip_reason" if the image should be excluded.
    """
    metadata = {
        "datetime": "", "camera_name": "", "shutter_speed": "",
        "weather": "unknown", "flash": False,
        "laplacian_var": -1.0,
        "skip": False, "skip_reason": "",
    }
    try:
        img_pil   = PILImage.open(image_path)
        exif_data = img_pil._getexif() if hasattr(img_pil, "_getexif") else dict(img_pil.getexif())  # type: ignore[attr-defined]
        if exif_data is None:
            return metadata
        tag_map = {TAGS.get(k, k): v for k, v in exif_data.items()}
        metadata["datetime"]    = str(tag_map.get("DateTimeOriginal", ""))
        metadata["camera_name"] = str(tag_map.get("Model", ""))

        # Weather from shutter speed
        exposure = exif_data.get(33434)  # ExposureTime
        if exposure:
            denom = exposure[1] if isinstance(exposure, tuple) else int(1 / exposure)
            metadata["shutter_speed"] = f"1/{denom}"
            metadata["weather"] = "sunny" if denom > cfg["sunny_shutter_threshold"] else "cloudy"

        # Flash detection — EXIF tag 37385
        # Value 0x0 = no flash, any nonzero value = flash fired (various modes)
        flash_val = exif_data.get(37385, 0)
        if flash_val and int(flash_val) != 0:
            metadata["flash"] = True
            if cfg.get("skip_flash", True):
                metadata["skip"]        = True
                metadata["skip_reason"] = f"flash (EXIF={flash_val})"

    except Exception:
        pass

    # Foggy/blurry detection — Laplacian variance on grayscale
    # Low variance = low contrast = fog, heavy cloud, or blur
    try:
        img = cv2.imread(str(image_path))
        if img is not None:
            gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
            lap_var = cv2.Laplacian(gray, cv2.CV_64F).var()
            metadata["laplacian_var"] = round(float(lap_var), 1)
            if not metadata["skip"] and cfg.get("skip_foggy", True):
                if lap_var < cfg.get("foggy_threshold", 50):
                    metadata["skip"]        = True
                    metadata["skip_reason"] = f"foggy/blurry (laplacian={lap_var:.1f})"
    except Exception:
        pass

    return metadata


# ══════════════════════════════════════════════════════════════════
# CLASSIFICATION HELPERS
# ══════════════════════════════════════════════════════════════════

def map_to_broad_class(order, cfg, common_name="", family=""):
    """
    Conservatively map InsectNet taxonomy to Arctic pollinator categories.

    Categories: bumblebee, fly, butterfly, other.

    - bumblebee: only when species/common name explicitly confirms it
    - fly: Diptera (includes hoverflies and muscid flies, dominant Arctic pollinators)
    - butterfly: Lepidoptera except moths
    - other: everything else, including Hymenoptera (wasps, ants, bees)
      Wasps are minor pollinators in Nordic/Arctic but too broad to map directly.
    """
    import math
    name = str(common_name).lower() if common_name and not (isinstance(common_name, float) and math.isnan(common_name)) else ""
    ord_ = str(order).lower() if order and not (isinstance(order, float) and math.isnan(order)) else ""

    # Bumblebee — only when species name explicitly confirms
    if "bumblebee" in name or "bumble bee" in name:
        return "bumblebee"

    # Fly — Diptera maps safely (includes hoverflies, muscid flies)
    if ord_ == "diptera":
        return "fly"

    # Butterfly — Lepidoptera, but exclude moths
    if ord_ == "lepidoptera":
        if "moth" in name or "hawk-moth" in name or "hawkmoth" in name:
            return "other"
        return "butterfly"

    # Hymenoptera (wasps, ants, bees) — too broad to map directly
    # Bumblebees already caught above via name check
    return "other"


def is_near_marked_flower(bbox, zone, cfg):
    """
    Returns True if the insect bounding box overlaps sufficiently with the ROI zone.
    Uses near_flower_iou_threshold as the minimum overlap fraction.
    """
    x, y, w, h = bbox
    roi_crop = zone[y:y+h, x:x+w]
    if roi_crop.size == 0:
        return False
    return np.count_nonzero(roi_crop) / roi_crop.size > cfg["near_flower_iou_threshold"]


# ══════════════════════════════════════════════════════════════════
# CSV
# ══════════════════════════════════════════════════════════════════

def init_csv(output_path, fields):
    """Create CSV file with headers. Overwrites existing file."""
    with open(output_path, "w", newline="") as f:
        csv.DictWriter(f, fieldnames=fields).writeheader()


def write_csv_row(output_path, row_dict, fields):
    """Append one row to the CSV. Missing fields are written as empty string."""
    with open(output_path, "a", newline="") as f:
        csv.DictWriter(f, fieldnames=fields).writerow(
            {k: row_dict.get(k, "") for k in fields}
        )


# ══════════════════════════════════════════════════════════════════
# DEBUG IMAGES
# ══════════════════════════════════════════════════════════════════

def save_debug_images(name, image, background, zone, marker, debug_dir, cfg):
    """Save debug images cropped to ROI bounding box for easy inspection."""
    # Get ROI bounding box — crop all debug images to this region
    coords = cv2.findNonZero(zone)
    if coords is not None:
        rx, ry, rw, rh = cv2.boundingRect(coords)
    else:
        rx, ry = 0, 0
        rh, rw = image.shape[:2]

    def roi_crop(img):
        """Crop image or mask to ROI bounding box."""
        return img[ry:ry+rh, rx:rx+rw] if len(img.shape) == 2 else img[ry:ry+rh, rx:rx+rw]

    # 1. Original image cropped to ROI — shows what the camera actually sees
    cv2.imwrite(str(debug_dir / f"{name}_1_original.jpg"), roi_crop(image))

    # 2. Difference mask — white = change detected within ROI
    gray_img = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    gray_bg  = cv2.cvtColor(background, cv2.COLOR_BGR2GRAY)
    gray_img = cv2.bitwise_and(gray_img, gray_img, mask=zone)
    gray_bg  = cv2.bitwise_and(gray_bg,  gray_bg,  mask=zone)
    diff     = cv2.absdiff(gray_bg, gray_img)
    diff     = cv2.GaussianBlur(diff, (7, 7), 0)
    _, mask  = cv2.threshold(diff, cfg["darker_threshold"], 255, cv2.THRESH_BINARY)
    mask     = cv2.bitwise_and(mask, zone)
    hsv      = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    mask     = cv2.bitwise_and(mask, cv2.bitwise_not(cv2.inRange(hsv, np.array([25,40,40]), np.array([95,255,255]))))
    kernel   = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    mask     = cv2.morphologyEx(cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel), cv2.MORPH_CLOSE, kernel)
    cv2.imwrite(str(debug_dir / f"{name}_2_diff.jpg"), roi_crop(mask))

    # 3. Contours cropped to ROI — green = passes filters, red = rejected
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    overlay2 = roi_crop(image.copy())
    for c in contours:
        area = cv2.contourArea(c)
        if area < 100:
            continue
        x, y, w, h = cv2.boundingRect(c)
        roi_px = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)[y:y+h, x:x+w]
        passes = (area >= cfg["min_contour_area"]
                  and area <= cfg["max_contour_area"]
                  and max(w,h)/max(min(w,h),1) <= cfg["max_aspect_ratio"])
        color = (0, 255, 0) if passes else (0, 0, 255)
        # Adjust coords to ROI-cropped image
        cv2.rectangle(overlay2, (x-rx, y-ry), (x-rx+w, y-ry+h), color, 2)
        cv2.putText(overlay2, f"a={int(area)} t={roi_px.std():.0f}", (x-rx, y-ry-10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)
    cv2.imwrite(str(debug_dir / f"{name}_3_contours.jpg"), overlay2)  # already roi_cropped


### Pipeline functions & Main

In [ ]:
def classify_and_write(paths, filtered_full, zone, model, cmn_df,
                       class_txt_path, output_csv, cfg, csv_fields,
                       crop_dir, debug=False, debug_dir=None, image_dir=None,):
    """
    Step 7: Classify all detections from full image and write to CSV.

    Each detection is checked against the ROI zone to set detection_scope:
      "roi"         — bbox overlaps with the marked flower zone
      "outside_roi" — bbox is elsewhere in the image
    Running one full-image detection pass is simpler than two separate passes.
    """
    init_csv(output_csv, csv_fields)

    total_paths = len(paths)
    for idx_p, path in enumerate(paths):
        # Use relative path as unique ID to avoid collision across subfolders
        if image_dir:
            try:
                rel_id = str(Path(path).relative_to(image_dir)).replace("/", "_").replace("\\", "_")
            except ValueError:
                rel_id = Path(path).name
        else:
            rel_id = Path(path).name
        path_str = str(path)
        image = cv2.imread(path_str)
        if image is None:
            continue

        metadata = get_exif_metadata(path_str, cfg)

        if metadata.get("skip"):
            print(f"  [{idx_p+1}/{total_paths}] {rel_id} | SKIP: {metadata['skip_reason']} | sharpness={metadata.get('laplacian_var',-1):.0f}")
            write_csv_row(output_csv, {
                "image_name":    rel_id,
                "datetime":      metadata["datetime"],
                "camera_name":   metadata["camera_name"],
                "shutter_speed": metadata["shutter_speed"],
                "weather":       metadata["weather"],
                "skip":          True,
                "skip_reason":   metadata["skip_reason"],
                "laplacian_var": metadata.get("laplacian_var", ""),
                "pollinator_detected": "skipped",
            }, csv_fields)
            continue

        detections = filtered_full.get(path_str, [])

        if not detections:
            print(f"  [{idx_p+1}/{total_paths}] {rel_id} | {metadata.get('weather','?')} | sharpness={metadata.get('laplacian_var',-1):.0f} | no detection")
            write_csv_row(output_csv,
                {"image_name": rel_id, **metadata, "pollinator_detected": "no"},
                csv_fields)
            continue

        print(f"  [{idx_p+1}/{total_paths}] {rel_id} | {metadata.get('weather','?')} | sharpness={metadata.get('laplacian_var',-1):.0f} | {len(detections)} candidate(s)")
        for i, bbox in enumerate(detections):
            x, y, w, h = bbox  # extract here so available everywhere
            crop     = crop_with_padding(image, bbox, cfg)
            crop_rgb = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)

            # Always save crop
            crop_filename = f"{rel_id.rsplit('.', 1)[0]}_crop_{i}.jpg"
            cv2.imwrite(str(crop_dir / crop_filename), crop)


            if cfg.get("skip_insectnet", False):
                write_csv_row(output_csv, {
                    "image_name":          rel_id,
                    "crop_filename":       crop_filename,
                    "pollinator_detected": "candidate",
                    "bbox_x": x, "bbox_y": y, "bbox_w": w, "bbox_h": h,
                    **metadata,
                }, csv_fields)
                continue

            try:
                sci, cmn, order, family, role, confirmed, other, confidence, energy = evaluate(
                    model, crop_rgb, cmn_df, class_txt_path
                )
            except (IndexError, KeyError) as e:
                # Species predicted by InsectNet not found in classes.csv
                sci, cmn, order, family, role = "", "", "", "", ""
                confirmed, other, confidence, energy = False, {}, 0.0, 999.0
            in_roi = is_near_marked_flower(bbox, zone, cfg)
            scope  = "roi" if in_roi else "outside_roi"

            if not confirmed:
                print(f"  {rel_id} [{scope}]: skipped — OOD (energy={energy:.2f})")
                write_csv_row(output_csv, {
                    "image_name": rel_id, **metadata,
                    "pollinator_detected": "uncertain",
                    "insectnet_confirmed": "False",
                    "scientific_name":     sci,
                    "detection_scope":     scope,
                    "near_marked_flower":  str(in_roi),
                    "confidence":          f"{confidence:.4f}",
                    "energy_score":        f"{energy:.4f}",
                    "bbox_x": x, "bbox_y": y, "bbox_w": w, "bbox_h": h,
                }, csv_fields)
                continue

            if role not in ["Pollinator", "Predator", "Parasitoid"]:
                print(f"  {rel_id} [{scope}]: skipped — role={role}")
                continue

            broad_class = map_to_broad_class(order, cfg, common_name=cmn, family=family)

            print(f"{rel_id} [{scope}]")
            print(f"  Weather:         {metadata['weather']} ({metadata['shutter_speed']})")
            print(f"  Broad class:     {broad_class}")
            print(f"  Scientific Name: {sci}")
            print(f"  Common Name:     {cmn}")
            print(f"  Order/Family:    {order} / {family}")
            print(f"  Confirmed:       {confirmed}  |  Confidence: {confidence:.1%}  |  Energy: {energy:.2f}")
            print(f"  Near flower:     {in_roi}  |  Scope: {scope}")
            if other:
                print(f"  Other plausible: {other}")
            print()

            write_csv_row(output_csv, {
                "image_name": rel_id, **metadata,
                "pollinator_detected": "yes",
                "pollinator_type":     broad_class,
                "scientific_name":     sci,
                "common_name":         cmn,
                "order":               order,
                "family":              family,
                "insectnet_confirmed": str(confirmed),
                "near_marked_flower":  str(in_roi),
                "detection_scope":     scope,
                "confidence":          f"{confidence:.4f}",
                "energy_score":        f"{energy:.4f}",
                "bbox_x": x, "bbox_y": y, "bbox_w": w, "bbox_h": h,
            }, csv_fields)


def main(image_dir, output_csv="results.csv", debug=False,
         manual_roi=None, config=None, debug_csv=False,
         crop_dir=None, debug_dir=None):
    """
    Main pipeline — orchestrates all steps.

    Args:
        image_dir:  path to folder containing images for one plot
        output_csv: path to output CSV file
        debug:      save intermediate debug images to debug_dir
        manual_roi: manually draw ROI instead of using green marker
        config:     dict of parameter overrides merged with DEFAULT_CONFIG
        debug_csv:  False = clean CSV for Maria (default)
                    True  = full debug CSV with bbox, confidence, energy
        crop_dir:   directory for detection crops (default: <output_csv parent>/crops)
        debug_dir:  directory for debug images  (default: <output_csv parent>/debug)
    """
    cfg        = {**DEFAULT_CONFIG, **(config or {})}
    csv_fields = CSV_FIELDS_DEBUG if debug_csv else CSV_FIELDS_MARIA

    image_dir  = Path(image_dir)
    output_csv = Path(output_csv)
    out_base  = output_csv.parent if output_csv.parent != Path('.') else PROJECT_DIR

    crop_dir  = Path(crop_dir)  if crop_dir  else out_base / "crops"
    debug_dir = Path(debug_dir) if debug_dir else out_base / "debug"
    crop_dir.mkdir(parents=True, exist_ok=True)

    # Sort by filename for time-order; use relative path as unique key to handle
    # duplicate filenames across subfolders
    all_found = list(image_dir.rglob("*.JPG")) + list(image_dir.rglob("*.jpg"))
    # Sort by EXIF DateTimeOriginal → filename number → filename string
    # This ensures correct temporal ordering for background subtraction
    # and frame-to-frame difference computation.
    paths = sorted(all_found, key=robust_sort_key)
    print(f"First 3 frames (sorted): {[p.name for p in paths[:3]]}")
    # Warn about duplicate filenames
    from collections import Counter
    name_counts = Counter(p.name for p in paths)
    dups = {k: v for k, v in name_counts.items() if v > 1}
    if dups:
        print(f"WARNING: {len(dups)} duplicate filename(s) found across subfolders.")
        print("Using relative path as unique ID in CSV and debug outputs.")
    if not paths:
        print(f"No images found in {image_dir}")
        return
    print(f"Found {len(paths)} images")

    if debug:
        debug_dir.mkdir(parents=True, exist_ok=True)
        print(f"Debug images will be saved to {debug_dir}/")

    # Use manual_roi arg if explicitly passed, otherwise fall back to config
    use_manual_roi = manual_roi if manual_roi is not None else cfg.get("manual_roi", False)
    zone, _ = setup_roi(paths, cfg, use_manual_roi)

    background = build_background(paths, cfg)
    if background is None and cfg.get("rolling_window", 0) == 0:
        print("Failed to build background.")
        return

    all_detections = detect_all_frames(paths, background, zone, cfg, debug, debug_dir, image_dir=image_dir)
    print(f"Before filter: {sum(len(v) for v in all_detections.values())} detections across {len(all_detections)} frames")
    filtered = all_detections  # static filter disabled
    print(f"After filter:  {sum(len(v) for v in filtered.values())} detections across {len(filtered)} frames")


    # Debug: check key format consistency
    if filtered:
        sample_key  = list(filtered.keys())[0]
        sample_path = str(paths[0])
        print(f"  filtered key : {sample_key!r}")
        print(f"  path_str     : {sample_path!r}")
        if sample_key != sample_path:
            print("  WARNING: key format mismatch — detections will not be found!")
        else:
            print("  Key format OK")
    else:
        print("  filtered is empty — no detections survived static filter")

    model, cmn_df, class_txt_path = load_model_and_classes()

    classify_and_write(
        paths, filtered, zone, model, cmn_df,
        class_txt_path, output_csv, cfg, csv_fields,
        crop_dir, debug, debug_dir, image_dir=image_dir,
    )
    print(f"\nDone. Results saved to {output_csv}")

### Batch Run

Process all leaf folders under a root directory. One CSV per folder saved to `results/`.

1. Set `ROOT` to your data directory
2. Run the preview cell to confirm folders
3. Run the execute cell to process all folders

### Run

Use default config, or override specific parameters via `config={}`. Only override what
need to change.

In [6]:
# ══════════════════════════════════════════════════════════════════
# BATCH RUN — one CSV per leaf folder
# ══════════════════════════════════════════════════════════════════
#
# A "leaf folder" is any directory that contains images and no subdirectories.
#
# Each leaf folder contains images from one camera at one fixed position
# covering one plot. Background subtraction is therefore valid within a
# single leaf folder, as the background is consistent across all frames.
#
# Folder structure example:
#   ROOT/
#   ├── hdd1_cg_asa_p1/
#   │   ├── 101_wsct/   ← one camera, one plot  →  hdd1_cg_asa_p1_101_wsct.csv
#   │   └── 102_wsct/   ← one camera, one plot  →  hdd1_cg_asa_p1_102_wsct.csv
#   └── hdd1_enbranten_aral_p1/
#       ├── 101_wsct/   ← one camera, one plot  →  hdd1_enbranten_aral_p1_101_wsct.csv
#       └── 102_wsct/   ← one camera, one plot  →  hdd1_enbranten_aral_p1_102_wsct.csv

from pathlib import Path

# ── Configure ─────────────────────────────────────────────────────
ROOT        = Path("Insects_images/Pollinators")  # CHANGE THIS

# Results saved alongside Pollinators folder (not inside it):
#   Insects_images/
#   ├── Pollinators/    ← image data (ROOT)
#   ├── results/        ← first run
#   ├── results_1/      ← second run (auto-incremented)
#   └── results_2/      ← third run
def make_results_dir(base: Path) -> Path:
    """Create results folder next to Pollinators, auto-increment if already exists."""
    candidate = base / "results"
    if not candidate.exists():
        candidate.mkdir(parents=True)
        return candidate
    i = 1
    while True:
        candidate = base / f"results_{i}"
        if not candidate.exists():
            candidate.mkdir(parents=True)
            return candidate
        i += 1

RESULTS_DIR = make_results_dir(ROOT.parent)
print(f"Results will be saved to: {RESULTS_DIR}")

BATCH_CONFIG = {
    "skip_insectnet": True,   # Set False for full pipeline
    "rolling_window": 1,
}

# ── Find all leaf folders ──────────────────────────────────────────
def get_leaf_dirs(root):
    """Return all leaf directories (no subdirectories) that contain images.

    Each leaf folder corresponds to one camera at one fixed position covering
    one plot. Images within a leaf folder share a consistent background,
    making background subtraction valid within each folder.
    """
    leaf = []
    for d in sorted(root.rglob("*")):
        if d.is_dir() and not any(x.is_dir() for x in d.iterdir()):
            if any(d.glob("*.JPG")) or any(d.glob("*.jpg")):
                leaf.append(d)
    return leaf

leaf_dirs = get_leaf_dirs(ROOT)

# ── Preview ───────────────────────────────────────────────────────
print(f"Found {len(leaf_dirs)} leaf folder(s):")
for d in leaf_dirs:
    n = len(list(d.glob("*.JPG"))) + len(list(d.glob("*.jpg")))
    csv_name = f"{d.parent.name}_{d.name}.csv"
    print(f"  {d.relative_to(ROOT)}  →  {n} images  →  {csv_name}")


Found 20 leaf folder(s):
  hdd1_cg_asa_p1/101_wsct  →  98 images  →  hdd1_cg_asa_p1_101_wsct.csv
  hdd1_cg_asa_p1/102_wsct  →  4 images  →  hdd1_cg_asa_p1_102_wsct.csv
  hdd1_cg_bal_p3_20250719/101_wsct  →  77 images  →  hdd1_cg_bal_p3_20250719_101_wsct.csv
  hdd1_cg_phyca_p1/101_wsct  →  67 images  →  hdd1_cg_phyca_p1_101_wsct.csv
  hdd1_cg_phyca_p1/102_wsct  →  36 images  →  hdd1_cg_phyca_p1_102_wsct.csv
  hdd1_cg_phyca_p1/103_wsct  →  9 images  →  hdd1_cg_phyca_p1_103_wsct.csv
  hdd1_cg_vamy_p1/101_wsct  →  11 images  →  hdd1_cg_vamy_p1_101_wsct.csv
  hdd1_cg_vau_p1_pela_p1/101_wsct  →  7 images  →  hdd1_cg_vau_p1_pela_p1_101_wsct.csv
  hdd1_cg_vau_p1_pela_p1/103_wsct  →  10 images  →  hdd1_cg_vau_p1_pela_p1_103_wsct.csv
  hdd1_cg_vau_p3_20250711/101_wsct  →  18 images  →  hdd1_cg_vau_p3_20250711_101_wsct.csv
  hdd1_cg_vau_p3_20250711/102_wsct  →  14 images  →  hdd1_cg_vau_p3_20250711_102_wsct.csv
  hdd1_enbranten_aral_p1/101_wsct  →  6 images  →  hdd1_enbranten_aral_p1_101_wsct.csv

In [7]:
# Default run
#main(
 #   image_dir  = PROJECT_DIR / "Insects_images",
  #  output_csv = "results.csv",
   # debug      = True,
    #manual_roi = False,
    #config = {
    #    "marker_hue":      (80, 100),
    #    "marker_sat_min":  100,
    #    "marker_val_min":  80,
    #    "marker_min_area": 50,
    #}
#)

# Example: override specific parameters after tuning
# main(
#     image_dir  = PROJECT_DIR / "images",
#     output_csv = "results.csv",
#     config = {
#         "sunny_shutter_threshold": 150,   # adjust after checking cloudy images
#         "static_max_frames":       10,    # tighten if too many false positives
#         "darker_threshold":        30,    # lower = more sensitive
#         "background_sample_size":  100,   # reduce if memory is tight
#     }
# )

In [8]:
# ── Run batch ─────────────────────────────────────────────────────
# Run the preview cell first to confirm the folder list looks correct.

for camera_dir in leaf_dirs:
    n = len(list(camera_dir.glob("*.JPG"))) + len(list(camera_dir.glob("*.jpg")))
    if n == 0:
        print(f"Skipping {camera_dir} — no images")
        continue

    # One output folder per camera: results/hdd1_cg_asa_p1_101_wsct/
    csv_name   = f"{camera_dir.parent.name}_{camera_dir.name}"
    camera_out = RESULTS_DIR / csv_name
    camera_out.mkdir(exist_ok=True)

    print(f"\n{'='*60}")
    print(f"Camera : {camera_dir.relative_to(ROOT)}")
    print(f"Images : {n}")
    print(f"Output : {camera_out}/")
    print(f"{'='*60}")

    try:
        main(
            image_dir  = camera_dir,
            output_csv = str(camera_out / "results.csv"),
            crop_dir   = str(camera_out / "crops"),
            debug_dir  = str(camera_out / "debug"),
            debug      = True,
            config     = BATCH_CONFIG,
        )
        print(f"\u2713 Done — {csv_name}/")
    except Exception as e:
        print(f"\u2717 ERROR in {camera_dir.name}: {e}")
        continue

print(f"\nAll done. CSVs saved under: {RESULTS_DIR}")



Camera : hdd1_cg_asa_p1/101_wsct
Images : 98
Output : Insects_images/Pollinators/results/hdd1_cg_asa_p1_101_wsct/
Found 98 images
Debug images will be saved to Insects_images/Pollinators/results/hdd1_cg_asa_p1_101_wsct/debug/
Marker found at (1365, 739)
Watching zone covers 39.0% of image
Background computed from 98 sampled frames (of 98 total)
Detecting visitors...
Before filter: 1989 detections across 98 frames
After filter:  1989 detections across 98 frames
  filtered key : 'Insects_images/Pollinators/hdd1_cg_asa_p1/101_wsct/WSCT6147.JPG'
  path_str     : 'Insects_images/Pollinators/hdd1_cg_asa_p1/101_wsct/WSCT6147.JPG'
  Key format OK
Loading model...
Model loaded.

  [1/98] WSCT6147.JPG | sunny | sharpness=473 | 5 candidate(s)
  [2/98] WSCT6148.JPG | sunny | sharpness=477 | 4 candidate(s)
  [3/98] WSCT6149.JPG | sunny | sharpness=479 | 9 candidate(s)
  [4/98] WSCT6844.JPG | sunny | sharpness=408 | 13 candidate(s)
  [5/98] WSCT6845.JPG | sunny | sharpness=405 | 2 candidate(s)
  [6